In [2]:
import sqlite3
# import pandas as pd

In [3]:
FILEPATH = "../data/rosbag2_2025_06_19-09_27_46_0.db3"
DIR = "../data/"

In [ ]:
# with sqlite3.connect(FILEPATH) as con:
#     cur = con.cursor()
#     cur.execute("SELECT name FROM sqlite_master WHERE type='table';")


#     tables = cur.fetchall()
#     print(f"Tables in the database: {tables}")

#     data = {}
#     for table in tables:
#         table_name = table[0]
#         print(f"Table: {table_name}")

#         data[table_name] = pd.read_sql_query(f"SELECT * FROM {table_name}", con)

In [5]:
# 1)  Built-in + custom type store
from rosbags.typesys import Stores, get_typestore, get_types_from_msg
ts = get_typestore(Stores.ROS2_HUMBLE)            # pick the distro that recorded the bag

# 2)  Paste the two message definitions (they’re only three lines total)
MSG_NAMEDPOSE = """
string name
geometry_msgs/Pose pose
"""

MSG_NAMEDPOSEARRAY = """
std_msgs/Header header
NamedPose[] poses
"""

# 3)  Register them under their fully-qualified names
ts.register(get_types_from_msg(MSG_NAMEDPOSE,
          'motion_capture_tracking_interfaces/msg/NamedPose'))
ts.register(get_types_from_msg(MSG_NAMEDPOSEARRAY,
          'motion_capture_tracking_interfaces/msg/NamedPoseArray'))

    

In [12]:
from pathlib import Path
from rosbags.highlevel import AnyReader
from datetime import datetime, timezone
from zoneinfo import ZoneInfo   # stdlib


bag_dir = Path(DIR)                           # folder with metadata.yaml

with AnyReader([bag_dir], default_typestore=ts) as reader:
    # 1) pick the connections whose topic you want
    mocap_conns = [c for c in reader.connections if c.topic == '/poses']

    # 2) iterate only those connections
    for conn, t_ns, raw in reader.messages(connections=mocap_conns):
        msg = reader.deserialize(raw, conn.msgtype)

        stamp = datetime.fromtimestamp(t_ns / 1e9, tz=timezone.utc).astimezone(ZoneInfo("Europe/Copenhagen"))


        # access the poses list
        print(f'{stamp}  {len(msg.poses)} poses  frame={msg.header.frame_id}')
        for p in msg.poses:
            pos = p.pose.position
            orient = p.pose.orientation
            print(f'  {p.name}: x={pos.x:.3f}, y={pos.y:.3f}, z={pos.z:.3f}, qx={orient.x:.3f}, qy={orient.y:.3f}, qz={orient.z:.3f}, qw={orient.w:.3f}')

2025-06-19 11:28:04.658487+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.703, z=0.668, qx=-0.799, qy=-0.285, qz=-0.499, qw=-0.181
2025-06-19 11:28:04.658661+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.703, z=0.668, qx=-0.799, qy=-0.285, qz=-0.499, qw=-0.181
2025-06-19 11:28:04.669129+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.704, z=0.669, qx=-0.795, qy=-0.287, qz=-0.504, qw=-0.177
2025-06-19 11:28:04.669173+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.704, z=0.669, qx=-0.795, qy=-0.287, qz=-0.504, qw=-0.177
2025-06-19 11:28:04.675328+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.705, z=0.670, qx=-0.794, qy=-0.292, qz=-0.504, qw=-0.175
2025-06-19 11:28:04.675490+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.705, z=0.670, qx=-0.794, qy=-0.292, qz=-0.504, qw=-0.175
2025-06-19 11:28:04.687746+02:00  1 poses  frame=world
  cool_drone: x=-4.070, y=0.705, z=0.670, qx=-0.793, qy=-0.292, qz=-0.507, qw=-0.171
2025-06-19 11:28:04.

In [11]:
orientation = p.pose.orientation
print(orientation)

geometry_msgs__msg__Quaternion(x=-0.7983046770095825, y=-0.2973348796367645, z=-0.4932419955730438, w=-0.17610768973827362, __msgtype__='geometry_msgs/msg/Quaternion')
